In [ ]:
# Google Colab setup
!pip install -q kailash-ml polars plotly gdown python-dotenv

# Mount Google Drive for datasets
from google.colab import drive
drive.mount("/content/drive")


# ════════════════════════════════════════════════════════════════════════
# ASCENT5 — Exercise 4: RAGResearchAgent
# ════════════════════════════════════════════════════════════════════════
# OBJECTIVE: Build RAGResearchAgent over Kailash SDK documentation and
#   Singapore regulatory docs. Evaluate retrieval quality.
#
# TASKS:
#   1. Prepare document corpus (SDK docs + regulatory)
#   2. Build RAGResearchAgent with document retrieval
#   3. Evaluate retrieval quality (faithfulness, relevance)
#   4. Compare dense vs hybrid retrieval
#   5. Demonstrate RAGAS evaluation metrics
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import os

import polars as pl
from dotenv import load_dotenv

from kaizen import Signature, InputField, OutputField
from kaizen_agents.agents.specialized.rag_research import RAGResearchAgent

from shared import ASCENTDataLoader
from shared.kailash_helpers import setup_environment

setup_environment()

model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))



## TASK 1: Prepare document corpus


In [ ]:
# Simulated document corpus (in production, load from ascent05/ data)
sdk_docs = [
    {
        "title": "DataExplorer — Automated Data Profiling",
        "content": "DataExplorer provides async profiling of polars DataFrames. Key methods: profile() returns DataProfile with column statistics, correlation matrices (Pearson, Spearman, Cramer's V), and data quality alerts. AlertConfig lets you configure thresholds for 8 alert types: high_nulls, constant, high_skewness, high_zeros, high_cardinality, high_correlation, duplicates, imbalanced.",
        "source": "kailash-ml docs",
    },
    {
        "title": "TrainingPipeline — Model Training Lifecycle",
        "content": "TrainingPipeline orchestrates model training with ModelSpec (model class + hyperparameters), EvalSpec (metrics, split strategy), and FeatureSchema. Supports sklearn, LightGBM, and PyTorch Lightning frameworks. Integrated with ExperimentTracker for logging and ModelRegistry for artifact storage.",
        "source": "kailash-ml docs",
    },
    # TODO: Add the remaining 4 documents to complete the corpus:
    #   - "DriftMonitor — Production Model Monitoring" (kailash-ml docs)
    #   - "Singapore AI Verify Framework" (IMDA Singapore)
    #   - "MAS Guidelines on AI in Financial Services" (MAS Guidelines)
    #   - "PACT Governance Framework" (kailash-pact docs)
    # Hint: Each dict has "title", "content", and "source" keys
    # Hint: DriftMonitor content covers PSI > 0.1 moderate drift, > 0.2 severe drift
    # Hint: AI Verify covers 11 principles including transparency, fairness, accountability
    # Hint: MAS FEAT covers fair, ethical, accountable, transparent for financial AI
    # Hint: PACT content covers D/T/R addressing and monotonic tightening
]

documents = [d["content"] for d in sdk_docs]
titles = [d["title"] for d in sdk_docs]

print(f"=== Document Corpus ===")
print(f"Documents: {len(documents)}")
for d in sdk_docs:
    print(f"  [{d['source']}] {d['title']}")



## TASK 2: Build RAGResearchAgent


RAG-based answer with source attribution.


In [ ]:
class RAGAnswer(Signature):

    # TODO: Define one InputField: question (user question)
    # Hint: question: str = InputField(description="____")
    question: str = ____

    # TODO: Define four OutputFields:
    #   answer (str grounded in retrieved documents),
    #   sources (list[str] of source document titles used),
    #   confidence (float 0-1),
    #   retrieval_quality (str assessing retrieval relevance)
    answer: str = ____
    sources: list[str] = ____
    confidence: float = ____
    retrieval_quality: str = ____


async def build_rag_agent():
    # TODO: Create a RAGResearchAgent with RAGAnswer signature,
    #       the documents list, and budget $3.00
    # Hint: RAGResearchAgent(signature=____, model=____, documents=____, max_llm_cost_usd=____)
    agent = ____

    questions = [
        "How does Kailash detect data drift in production models?",
        "What does Singapore's AI Verify framework require for financial AI?",
        "How does PACT ensure agents cannot exceed their governance boundaries?",
    ]

    results = []
    for q in questions:
        # TODO: Run the agent passing question as a keyword argument
        # Hint: await agent.run(question=____)
        result = ____

        print(f"\n=== Q: {q} ===")
        print(f"A: {result.answer[:300]}...")
        print(f"Sources: {result.sources}")
        print(f"Confidence: {result.confidence}")
        print(f"Retrieval quality: {result.retrieval_quality}")
        results.append(result)

    return results


rag_results  = await build_rag_agent()



## TASK 3: Evaluate retrieval quality (RAGAS-style)


Are all claims in the answer supported by the context?


Does the answer address the question?


In [ ]:
# RAGAS metrics (simplified implementation)
def evaluate_faithfulness(answer: str, context: str) -> float:
    # TODO: Split answer into sentences (on "."), filter to those with >10 chars
    # Hint: answer_sentences = [s.strip() for s in answer.split("____") if len(s.strip()) > ____]
    # TODO: Count how many sentences have at least one long word (>4 chars) present in context
    # Hint: any(word.lower() in context.lower() for word in s.split() if len(word) > 4)
    # TODO: Return supported / len(answer_sentences); return 1.0 if no sentences
    ____


def evaluate_relevance(question: str, answer: str) -> float:
    # TODO: Compute word overlap between question and answer (both lowercased, split on spaces)
    # Hint: q_words = set(question.lower().split())
    #       a_words = set(answer.lower().split())
    #       overlap = len(q_words & a_words) / max(len(q_words), 1)
    # TODO: Return min(overlap * 2, 1.0) to scale up the score
    ____


print(f"\n=== RAGAS-style Evaluation ===")
questions = [
    "How does Kailash detect data drift in production models?",
    "What does Singapore's AI Verify framework require for financial AI?",
    "How does PACT ensure agents cannot exceed their governance boundaries?",
]

for i, (q, r) in enumerate(zip(questions, rag_results)):
    relevant_docs = " ".join(
        d["content"] for d in sdk_docs if any(t in d["title"] for t in r.sources)
    )
    faith = evaluate_faithfulness(r.answer, relevant_docs) if relevant_docs else 0.0
    relev = evaluate_relevance(q, r.answer)

    print(f"\nQ{i+1}: {q[:60]}...")
    print(f"  Faithfulness: {faith:.3f}")
    print(f"  Relevance:    {relev:.3f}")
    print(f"  Self-rated confidence: {r.confidence:.3f}")



## TASK 4: Retrieval strategy comparison


Dense retrieval (embedding similarity):
  + Captures semantic meaning ("data drift" matches "distribution shift")
  - Expensive to compute, requires embedding model
  - Can hallucinate relevance for semantically similar but factually unrelated docs

Sparse retrieval (BM25/TF-IDF):
  + Fast, exact keyword matching
  + No embedding model needed
  - Misses synonyms ("drift" won't match "shift")
  - No semantic understanding

Hybrid (dense + sparse):
  + Best of both worlds
  + Re-ranking with cross-encoder for precision
  - Most complex, highest latency
  → This is what production RAG systems use


In [ ]:
print(f"\n=== Retrieval Strategy Comparison ===")
print(
)



## TASK 5: RAGAS metrics explained


1. Faithfulness: Are claims in the answer grounded in retrieved context?
   - Low faithfulness = model is hallucinating beyond the documents
   - Fix: better retrieval, stricter prompting, Corrective RAG

2. Answer Relevance: Does the answer actually address the question?
   - Low relevance = model is regurgitating context without answering
   - Fix: better question-answer alignment in the prompt

3. Context Precision: Is the retrieved context relevant to the question?
   - Low precision = retriever is pulling irrelevant documents
   - Fix: better embeddings, re-ranking, hybrid retrieval

4. Context Recall: Does the retrieved context cover all needed info?
   - Low recall = important documents are being missed
   - Fix: more documents, better chunking, parent-document retrieval

Your RAG system is only as good as its weakest metric.
High relevance + low faithfulness = confident hallucination (dangerous!)


In [ ]:
print(f"\n=== RAGAS Framework (Industry Standard) ===")
print(
)

print("\n✓ Exercise 4 complete — RAGResearchAgent with retrieval evaluation")

